# G6M2: Baseline vs Adaptive Window — Unified Comparator

Compare two Urc1 variants:
- **Baseline**: Fixed sliding window (standard Urc1)
- **Adaptive**: Greedy window expansion with freshness/physics gates (`Urc1_Adaptive`)

Goals:
- Use `UnifiedModelComparator` for systematic comparison across reference currents.
- Provide cross-model diagnostics (fit quality, coefficient evolution, coverage).

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

from degradation_toolbox.Urc.Urc1 import Urc1
from degradation_toolbox.Urc.Urc1_adaptive import Urc1_Adaptive
from master_arbeit_Di.explore.UnifiedModelComparator import UnifiedModelComparator
from master_arbeit_Di.explore.GMpreprocess import GMpreprocess

In [2]:
# =============================================================================
# CONFIGURATION SECTION - EDIT THESE TO CUSTOMIZE YOUR ANALYSIS
# =============================================================================
# Dataset and directories
DATASET_PATH = r"..\\..\\explore_data\\G6M2.parquet"
PREPROCESS_OUTPUT_DIR = r"..\\..\\explore_data\\output"
PLOTS_OUTPUT_DIR = r"..\\plots\\adaptive\\G6M2_comparison"
SAVE_PLOTS = True

# Reference condition configurations: Low, Medium, High
REF_CONFIGS = {
    "Low": {
        "Iref": 0.28,
        "Tref": 57,
        "OHref": 10,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_low_load__iref_0p28__tref_57__ohref_10__daily_regression_full_coverage.csv",
    },
    "Medium": {
        "Iref": 1.0,
        "Tref": 58,
        "OHref": 33,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_mid_load__iref_1__tref_58__ohref_33__daily_regression_full_coverage.csv",
    },
    "High": {
        "Iref": 1.31,
        "Tref": 58,
        "OHref": 18,
        "gt_file": r"..\\ground_truth\\output_backup\\gt_raw_processed\\G6M2__gt_diagram__g6m2_high_load__iref_1p31__tref_58__ohref_18__daily_regression_full_coverage.csv",
    },
}

# Shared model config
COMMON_CONFIG = {
    "Iref": [cfg["Iref"] for cfg in REF_CONFIGS.values()],
    "Tref": 60,
    "OHref": 72,
    "ref_config": REF_CONFIGS,
    "len_interval": 1,
    "slide": 1,
    "min_num_data_required_for_fit": 300,
    "threshold": 1e6,
    "i_off": 0.1,
    "u_off": 1.3,
    "plot_fit": 0,
    "data_filter_i_min": 0.1,
    "data_filter_U_min": 1.4,
    "data_filter_U_max": 2.3,
    "data_filter_T_min": 50,
    "data_filter_T_max": 65,
}

# Adaptive-specific config
ADAPTIVE_CONFIG = {
    "initial_window_size": 5,
    "min_target_count_ratio": 0.1,
    "fixed_freshness_threshold": 0.2,
    "decay_rate": 0.2,
    "urc_min": 1.4,
    "urc_max": 2.4,
}

# Comparison settings
SHOW_GT_METRICS = True
SHOW_ALL_COND_METRICS = True
REF_ORDER = list(REF_CONFIGS.keys())

In [3]:
# =============================================================================
# 1. DATA LOADING & PREPROCESSING
# =============================================================================
print("=" * 80)
print("STEP 1: Data Loading & Preprocessing")
print("=" * 80)

preprocessor = GMpreprocess(file_path=DATASET_PATH, output_dir=PREPROCESS_OUTPUT_DIR)
data = preprocessor.run()
dataset_name = preprocessor.name

print(f"Dataset: {dataset_name}, shape: {data.shape}")
print(f"Time range: {data.index.min()} -> {data.index.max()}\n")

shared_pre = Urc1.preprocess_once(
    data,
    i_off=COMMON_CONFIG["i_off"],
    u_off=COMMON_CONFIG["u_off"],
    data_filter_i_min=COMMON_CONFIG["data_filter_i_min"],
    data_filter_U_min=COMMON_CONFIG["data_filter_U_min"],
    data_filter_U_max=COMMON_CONFIG["data_filter_U_max"],
    data_filter_T_min=COMMON_CONFIG["data_filter_T_min"],
    data_filter_T_max=COMMON_CONFIG["data_filter_T_max"],
)
print(f"Shared preprocessed rows: {len(shared_pre)}\n")

STEP 1: Data Loading & Preprocessing
=== 1. Loading & Preprocessing: G6M2 ===
>> Data loaded successfully.
   [Detected] Temperature Col: 'Temp_Module_12' -> ID: '2'
>> Columns renamed to standard format.
>> Columns filtered. Retained: ['currentDensity', 'temperature', 'voltage']

=== 3. Saving Preprocessd data in .parquet format ===
>> ✅ Final Results saved successfully to:
   ..\\..\\explore_data\\output\G6M2_20260502_123400.parquet

=== GMpreprocess Pipeline Completed Successfully ===
Dataset: G6M2, shape: (1096020, 3)
Time range: 2023-07-06 00:00:00 -> 2025-08-05 09:29:00

[preprocess_once] 1096020 -> 359353 points.
Shared preprocessed rows: 359353



In [4]:
# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
print("=" * 80)
print("STEP 2: Model Training")
print("=" * 80)

models = {}

# ── 2a. Baseline ──
print("\n  -> Training Baseline (Urc1) model...")
urc_baseline = Urc1(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
)
models["Baseline"] = urc_baseline
print("  OK Baseline training completed")

# ── 2b. Adaptive ──
print("\n  -> Training Adaptive (Urc1_Adaptive) model...")
urc_adaptive = Urc1_Adaptive(
    data=data,
    name=dataset_name,
    preprocessed_data=shared_pre,
    **COMMON_CONFIG,
    **ADAPTIVE_CONFIG,
)
models["Adaptive"] = urc_adaptive
print("  OK Adaptive training completed")

print(f"\nOK {len(models)} models trained successfully")
print(f"\nAdaptive settings:")
for k, v in ADAPTIVE_CONFIG.items():
    print(f"  - {k}: {v}")

STEP 2: Model Training

  -> Training Baseline (Urc1) model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Fitting Stats: 323 intervals low data, 0 fit failed.
350 out of 762 fitting results are reliable.
  OK Baseline training completed

  -> Training Adaptive (Urc1_Adaptive) model...
Using shared preprocessed data (359353 points, skipping preprocess).
Voltage model fitting ...
Starting Adaptive Fitting for approx 757 intervals...
294 out of 762 fitting results are reliable.
Initialized Urc1_Adaptive (Simplified): Init_Win=5d
  Filters: Target_Min_Ratio=0.1, Freshness=0.2
  OK Adaptive training completed

OK 2 models trained successfully

Adaptive settings:
  - initial_window_size: 5
  - min_target_count_ratio: 0.1
  - fixed_freshness_threshold: 0.2
  - decay_rate: 0.2
  - urc_min: 1.4
  - urc_max: 2.4


In [5]:
# =============================================================================
# 3. COMPARATOR SETUP & GT LOADING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Initialize Comparator & Load Ground Truth")
print("=" * 80)

comparator = UnifiedModelComparator(models)
print(f"OK UnifiedModelComparator initialized with {len(models)} models\n")

gt_loaded_count = 0
for ref_name, ref_cfg in REF_CONFIGS.items():
    iref = ref_cfg["Iref"]
    gt_file = ref_cfg["gt_file"]
    gt_path = Path(gt_file)
    if not gt_path.is_absolute():
        gt_path = Path.cwd() / gt_path
    gt_path = gt_path.resolve()

    if gt_path.exists():
        try:
            gt_data = pd.read_csv(gt_path, index_col=0, parse_dates=True)
            colmap = {str(c).strip().lower(): c for c in gt_data.columns}
            candidate_cols = ["gt_uref_regression", "voltage", "uref", "gt_uref"]
            selected_col = None
            for c in candidate_cols:
                if c in colmap:
                    selected_col = colmap[c]
                    break

            if selected_col is not None:
                gt_series = gt_data[selected_col]
            elif len(gt_data.columns) == 1:
                gt_series = gt_data.iloc[:, 0]
                selected_col = gt_data.columns[0]
            else:
                raise ValueError(
                    f"Cannot identify voltage column: {gt_data.columns.tolist()}"
                )

            gt_series = gt_series.dropna()
            comparator.set_ground_truth(gt_series, iref=iref)
            gt_loaded_count += 1
            print(
                f"  OK {ref_name} (Iref={iref}) GT loaded: {len(gt_series)} points [column: {selected_col}]"
            )
        except Exception as e:
            print(f"  FAILED {ref_name} GT: {e}")
    else:
        print(f"  MISSING GT file: {gt_path}")

has_gt = gt_loaded_count > 0 and SHOW_GT_METRICS
print(f"\nGT status: {gt_loaded_count}/{len(REF_CONFIGS)} references loaded")
print(f"GT metrics enabled: {has_gt}\n")


STEP 3: Initialize Comparator & Load Ground Truth
OK UnifiedModelComparator initialized with 2 models

  OK Low (Iref=0.28) GT loaded: 758 points [column: gt_uref_regression]
  OK Medium (Iref=1.0) GT loaded: 758 points [column: gt_uref_regression]
  OK High (Iref=1.31) GT loaded: 758 points [column: gt_uref_regression]

GT status: 3/3 references loaded
GT metrics enabled: True



In [6]:
# =============================================================================
# 4. REFERENCE-SPECIFIC ANALYSIS (UNIFIED COMPARATOR)
# =============================================================================
print("=" * 80)
print("STEP 4: Reference-Specific Metrics & Comparison")
print("=" * 80)

rate_tables = []
for ref_name in REF_ORDER:
    ref_cfg = REF_CONFIGS[ref_name]
    iref = ref_cfg["Iref"]
    tref = ref_cfg["Tref"]
    ohref = ref_cfg["OHref"]

    print(f"\n### REFERENCE CONDITION: {ref_name} (Iref={iref}, Tref={tref}, OHref={ohref})")

    df_metrics = comparator.compare_all(
        i_target=iref,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
        include_gt_metrics=has_gt,
    )

    if df_metrics.empty:
        print("No metrics returned for this reference.")
        continue

    display(Markdown(df_metrics.to_markdown(index=False)))

    rate_tables.append(
        df_metrics[["Model Name", "Target Current (A/cm2)", "Degradation Rate (uV/h)", "Slope Sigma (uV/h)"]].assign(
            Reference=ref_name
        )
    )

    try:
        comparator.plot_interactive_trends(
            target_i=iref,
            show_gt=has_gt,
            save=SAVE_PLOTS,
            output_dir=PLOTS_OUTPUT_DIR,
        )
        print("OK Trend plot finished")
    except Exception as e:
        print(f"Trend plot failed: {e}")

    comparator.print_comparison_report(
        i_target=iref,
        include_gt_metrics=has_gt,
        include_all_cond_metrics=SHOW_ALL_COND_METRICS,
    )

STEP 4: Reference-Specific Metrics & Comparison

### REFERENCE CONDITION: Low (Iref=0.28, Tref=57, OHref=10)


| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                     0.28 |              5.331 |               350 |                   2.06814 |       7.677 |                    0 |               0.907 |               9 |          2.57 |              87.016 |          1.687 |               5.184 |           14.312 |                       435 | all_fitted   |         10.043 |         7.482 |                  350 |
| Adaptive     |                     0.28 |              9.407 |               294 |                   2.21699 |       2.52  |                    0 |               0.987 |              13 |          4.42 |               9.172 |          2.087 |               4.93  |            5.439 |                       298 | all_fitted   |          8.29  |         7.67  |                  294 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 0.28 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      350
  Deg Rate:         2.068140 μV/h
  RMSE:             7.677 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.907
  Outliers:         9 (2.6%)
  Max Residual:     87.016 mV
  Mean SE:          1.687 mV
  Cond (Median):    10^5.2
  Cond Scope:       all_fitted (n=435)
  GT RMSE:          10.043 mV (n=350)
  GT MAE:           7.482 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      294
  Deg Rate:         2.216991 μV/h
  RMSE:             2.520 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.987
  Outliers:         13 (4.4%)
  Max Residual:     9.172 mV
  Mean SE:          2.087 mV
  Cond (Median):    10^4.9
  Cond Scope:       all_fitted (n=298)
  GT RMSE:          8.290 mV (n=294)
  GT MAE:           7.670 mV

🏆 Best RMSE (vs model data):    Adaptive
🏆 Best Monotonicity: 

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                        1 |              5.331 |               350 |                   5.74563 |       8.044 |                    0 |               0.972 |              13 |          3.71 |              52.413 |          1.71  |               5.184 |           14.312 |                       435 | all_fitted   |         15.586 |        13.537 |                  350 |
| Adaptive     |                        1 |              9.407 |               294 |                   5.80673 |       6.031 |                    0 |               0.989 |              11 |          3.74 |              16.147 |          2.109 |               4.93  |            5.439 |                       298 | all_fitted   |         13.904 |        12.59  |                  294 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.0 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      350
  Deg Rate:         5.745628 μV/h
  RMSE:             8.044 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.972
  Outliers:         13 (3.7%)
  Max Residual:     52.413 mV
  Mean SE:          1.710 mV
  Cond (Median):    10^5.2
  Cond Scope:       all_fitted (n=435)
  GT RMSE:          15.586 mV (n=350)
  GT MAE:           13.537 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      294
  Deg Rate:         5.806726 μV/h
  RMSE:             6.031 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.989
  Outliers:         11 (3.7%)
  Max Residual:     16.147 mV
  Mean SE:          2.109 mV
  Cond (Median):    10^4.9
  Cond Scope:       all_fitted (n=298)
  GT RMSE:          13.904 mV (n=294)
  GT MAE:           12.590 mV

🏆 Best RMSE (vs model data):    Adaptive
🏆 Best Monotonici

| Model Name   |   Target Current (A/cm2) |   Fitting Time (s) |   Data Points (n) |   Degradation Rate (uV/h) |   RMSE (mV) |   Slope Sigma (uV/h) |   Mono (Rank) [0-1] |   Outlier Count |   Outlier (%) |   Max Residual (mV) |   Mean SE (mV) |   Cond Median log10 |   Cond P95 log10 |   Cond Count (Used Scope) | Cond Scope   |   GT RMSE (mV) |   GT MAE (mV) |   GT Valid Intervals |
|:-------------|-------------------------:|-------------------:|------------------:|--------------------------:|------------:|---------------------:|--------------------:|----------------:|--------------:|--------------------:|---------------:|--------------------:|-----------------:|--------------------------:|:-------------|---------------:|--------------:|---------------------:|
| Baseline     |                     1.31 |              5.331 |               350 |                   8.36521 |      12.103 |                    0 |               0.971 |              14 |           4   |              71.629 |          2.25  |               5.184 |           14.312 |                       435 | all_fitted   |         15.687 |        11.931 |                  350 |
| Adaptive     |                     1.31 |              9.407 |               294 |                   8.36254 |       8.938 |                    0 |               0.989 |               5 |           1.7 |              22.289 |          2.311 |               4.93  |            5.439 |                       298 | all_fitted   |         12.351 |         9.493 |                  294 |

OK Trend plot finished
  MODEL COMPARISON REPORT @ 1.31 A/cm²

📊 Baseline
------------------------------------------------------------
  Data Points:      350
  Deg Rate:         8.365210 μV/h
  RMSE:             12.103 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.971
  Outliers:         14 (4.0%)
  Max Residual:     71.629 mV
  Mean SE:          2.250 mV
  Cond (Median):    10^5.2
  Cond Scope:       all_fitted (n=435)
  GT RMSE:          15.687 mV (n=350)
  GT MAE:           11.931 mV

📊 Adaptive
------------------------------------------------------------
  Data Points:      294
  Deg Rate:         8.362544 μV/h
  RMSE:             8.938 mV
  Slope Sigma:      0.000 μV/h
  Mono (Rank):      0.989
  Outliers:         5 (1.7%)
  Max Residual:     22.289 mV
  Mean SE:          2.311 mV
  Cond (Median):    10^4.9
  Cond Scope:       all_fitted (n=298)
  GT RMSE:          12.351 mV (n=294)
  GT MAE:           9.493 mV

🏆 Best RMSE (vs model data):    Adaptive
🏆 Best Monotonici

In [7]:
# =============================================================================
# 5. CROSS-MODEL DIAGNOSTICS + SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Cross-Model Diagnostics")
print("=" * 80)

try:
    print("\n-> Plotting fit quality (RMSE & R2 distributions)...")
    comparator.plot_fit_quality(save=SAVE_PLOTS)
    print("OK Fit quality completed")
except Exception as e:
    print(f"Fit quality plot failed: {e}")

try:
    print("\n-> Plotting coefficient diagnostics...")
    comparator.plot_coefficient_diagnostic(save=SAVE_PLOTS, include_c6=False)
    print("OK Coefficient diagnostic completed")
except Exception as e:
    print(f"Coefficient diagnostic failed: {e}")

try:
    print("\n-> Plotting coverage Gantt...")
    comparator.plot_coverage_gantt(save=SAVE_PLOTS)
    print("OK Coverage Gantt completed")
except Exception as e:
    print(f"Coverage Gantt failed: {e}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)
print(f"\nModels trained: {len(models)}")
print(f"Reference conditions analyzed: {len(REF_CONFIGS)}")
print(f"Ground truth loaded: {has_gt}")
print(f"Output plots dir: {PLOTS_OUTPUT_DIR}" if SAVE_PLOTS else "Plots displayed only")
print("\nAdaptive settings:")
for k, v in ADAPTIVE_CONFIG.items():
    print(f"  - {k}: {v}")

if rate_tables:
    print("\nDegradation-rate summary across references:")
    display(pd.concat(rate_tables, ignore_index=True))


STEP 5: Cross-Model Diagnostics

-> Plotting fit quality (RMSE & R2 distributions)...
OK Fit quality completed

-> Plotting coefficient diagnostics...
OK Coefficient diagnostic completed

-> Plotting coverage Gantt...
OK Coverage Gantt completed

ANALYSIS COMPLETE

Models trained: 2
Reference conditions analyzed: 3
Ground truth loaded: True
Output plots dir: ..\\plots\\adaptive\\G6M2_comparison

Adaptive settings:
  - initial_window_size: 5
  - min_target_count_ratio: 0.1
  - fixed_freshness_threshold: 0.2
  - decay_rate: 0.2
  - urc_min: 1.4
  - urc_max: 2.4

Degradation-rate summary across references:


,Model Name,Target Current (A/cm2),Degradation Rate (uV/h),Slope Sigma (uV/h),Reference
0,Baseline,0.28,2.068140,0.0,Low
1,Adaptive,0.28,2.216991,0.0,Low
2,Baseline,1.00,5.745628,0.0,Medium
3,Adaptive,1.00,5.806726,0.0,Medium
4,Baseline,1.31,8.365210,0.0,High
5,Adaptive,1.31,8.362544,0.0,High
